# DCGAN Multimodal — Dataset de Animación

## Descripción del Dataset

Este cuadernillo implementa una **DCGAN (Deep Convolutional GAN)** adaptada para el **Conjunto de datos multimodales para el análisis de contenido animado**, que incluye:

| Modalidad | Descripción |
|-----------|-------------|
| 📊 **Tabular** (CSV) | Métricas sintéticas de participación, diseño y rendimiento |
| 🖼️ **Imágenes** (.npy) | Matrices en escala de grises de **64×64** píxeles |
| 🔊 **Audio** (.wav) | Formas de onda de audio simuladas (≥ 3 seg) |

- **Filas:** 1,258 muestras  
- **Columna objetivo:** etiqueta binaria (0 ó 1)
- **Licencia:** CC0 — Dominio público

### Objetivo

Entrenar una DCGAN que **genere nuevas imágenes de storyboard de animación** de 64×64 px a partir de ruido aleatorio, usando la modalidad de imágenes del dataset multimodal. La arquitectura convolucional permite capturar patrones espaciales que una GAN simple (MLP) no puede aprender correctamente.

## 1. Importaciones y configuración del dispositivo

In [6]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy.io.wavfile as wavfile
import warnings
warnings.filterwarnings('ignore')

# Reproducibilidad
torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Dispositivo: cuda
GPU: NVIDIA GeForce RTX 2050


## 2. Rutas del dataset multimodal

In [9]:
# ── RUTA CORREGIDA SEGÚN TU IMAGEN ──
ruta_imagenes = r'D:\2026\IA-2026-DATASETS\images'

## 3. Exploración del dataset multimodal

### 3.1 Datos tabulares (CSV)

In [10]:
# 1. Verificar si la carpeta existe
if os.path.exists(ruta_imagenes):
    print(f"[✓] Carpeta encontrada: {ruta_imagenes}")
else:
    print(f"[✗] ERROR: La carpeta NO existe.")

[✓] Carpeta encontrada: D:\2026\IA-2026-DATASETS\images


In [11]:
# 2. Verificar si hay archivos .npy dentro de esa carpeta
archivos_npy = glob.glob(os.path.join(ruta_imagenes, '*.npy'))
cantidad_imagenes = len(archivos_npy)

if cantidad_imagenes > 0:
    print(f"[✓] ¡Éxito! Se encontraron {cantidad_imagenes} imágenes .npy listas para procesar.")
else:
    print("[✗] ERROR: La carpeta existe, pero no hay archivos .npy dentro. Revisa si están en otra subcarpeta.")

[✓] ¡Éxito! Se encontraron 1258 imágenes .npy listas para procesar.


In [ ]:
# Distribución de la variable objetivo
if 'target' in df.columns or 'label' in df.columns:
    col_target = 'target' if 'target' in df.columns else 'label'
    conteos = df[col_target].value_counts()
    print(f"Distribución de clases en '{col_target}':")
    print(conteos)
    conteos.plot(kind='bar', title='Distribución de clases', color=['steelblue','salmon'])
    plt.xlabel('Clase'); plt.ylabel('Cantidad'); plt.tight_layout(); plt.show()
else:
    # Buscar columna binaria
    for col in df.columns:
        if df[col].nunique() == 2:
            print(f"Posible columna objetivo: '{col}'")
            print(df[col].value_counts())
            break

NameError: name 'df' is not defined

### 3.2 Datos de imagen (.npy — 64×64)

In [ ]:
# Listar archivos de imagen
archivos_img = sorted(glob.glob(os.path.join(ruta_imagenes, '*.npy')))
print(f"Total de imágenes (.npy): {len(archivos_img)}")

# Inspeccionar la primera imagen
muestra_img = np.load(archivos_img[0])
print(f"Forma de la imagen: {muestra_img.shape}")
print(f"Tipo de dato: {muestra_img.dtype}")
print(f"Rango de valores: [{muestra_img.min():.3f}, {muestra_img.max():.3f}]")

In [ ]:
# Visualizar una muestra de imágenes reales
n_show = 15
indices = np.random.choice(len(archivos_img), n_show, replace=False)

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
fig.suptitle('Muestra de imágenes reales del dataset (64×64, escala de grises)', fontsize=14)
for ax, idx in zip(axes.flatten(), indices):
    img = np.load(archivos_img[idx])
    if img.ndim == 3:
        img = img[:, :, 0]  # tomar primer canal si tiene más
    ax.imshow(img, cmap='gray')
    ax.axis('off')
    ax.set_title(f'ID {idx}', fontsize=8)
plt.tight_layout()
plt.show()

### 3.3 Datos de audio (.wav)

In [ ]:
archivos_wav = sorted(glob.glob(os.path.join(ruta_audio, '*.wav')))
print(f"Total de archivos de audio (.wav): {len(archivos_wav)}")

if archivos_wav:
    sr, datos_audio = wavfile.read(archivos_wav[0])
    duracion = len(datos_audio) / sr
    print(f"\nMuestra de audio [0]:")
    print(f"  Sample rate : {sr} Hz")
    print(f"  Muestras    : {len(datos_audio)}")
    print(f"  Duración    : {duracion:.2f} seg")
    print(f"  Canales     : {datos_audio.ndim}")
    print(f"  Tipo        : {datos_audio.dtype}")
    
    # Visualizar forma de onda
    t = np.linspace(0, duracion, len(datos_audio))
    plt.figure(figsize=(10, 3))
    plt.plot(t, datos_audio if datos_audio.ndim == 1 else datos_audio[:, 0], 
             alpha=0.7, color='steelblue')
    plt.title('Forma de onda del audio (muestra)')
    plt.xlabel('Tiempo (s)'); plt.ylabel('Amplitud')
    plt.tight_layout(); plt.show()

## 4. Dataset de PyTorch — Carga de imágenes multimodal

In [ ]:
class DatasetAnimacion(torch.utils.data.Dataset):
    """
    Dataset multimodal para el dataset de animación.
    
    Carga imágenes .npy de 64×64 en escala de grises.
    Los valores se normalizan al rango [-1, 1] para usarlos
    con la activación Tanh del generador.
    """
    
    def __init__(self, ruta_imagenes, ruta_csv=None):
        self.archivos = sorted(glob.glob(os.path.join(ruta_imagenes, '*.npy')))
        if not self.archivos:
            raise FileNotFoundError(f"No se encontraron archivos .npy en: {ruta_imagenes}")
        
        # Cargar metadatos tabulares opcionales
        self.df = pd.read_csv(ruta_csv) if (ruta_csv and os.path.exists(ruta_csv)) else None
        
        # Pre-cargar todas las imágenes en memoria para mayor velocidad
        print(f"Cargando {len(self.archivos)} imágenes en memoria...")
        imagenes = []
        for ruta in self.archivos:
            img = np.load(ruta).astype(np.float32)
            # Asegurar shape (H, W)
            if img.ndim == 3:
                img = img[:, :, 0]
            # Normalizar a [0,1] si está en [0,255]
            if img.max() > 1.0:
                img = img / 255.0
            # Normalizar a [-1, 1] para Tanh
            img = img * 2.0 - 1.0
            imagenes.append(img)
        
        # Shape final: (N, 1, 64, 64) — tensor en CUDA si disponible
        arr = np.stack(imagenes, axis=0)[:, np.newaxis, :, :]  # (N, 1, H, W)
        self.imagenes = torch.tensor(arr, dtype=torch.float32, device=device)
        print(f"Cargado: {self.imagenes.shape} | rango [{self.imagenes.min():.2f}, {self.imagenes.max():.2f}]")
    
    def __len__(self):
        return len(self.imagenes)
    
    def __getitem__(self, idx):
        return self.imagenes[idx], 0  # (imagen, etiqueta_dummy)


# Instanciar dataset
dataset = DatasetAnimacion(ruta_imagenes, ruta_csv)
print(f"\nDataset creado: {len(dataset)} muestras")

In [ ]:
# DataLoader
BATCH_SIZE = 32

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

imgs_batch, _ = next(iter(dataloader))
print(f"Forma del batch: {imgs_batch.shape}")
print(f"  dtype : {imgs_batch.dtype}")
print(f"  rango : [{imgs_batch.min():.2f}, {imgs_batch.max():.2f}]")

In [ ]:
# Verificar visualmente las imágenes normalizadas
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Imágenes reales normalizadas (rango [-1,1] → visualizadas)', fontsize=12)
for i, ax in enumerate(axes.flatten()):
    img = imgs_batch[i, 0].cpu().numpy()  # (64, 64)
    ax.imshow(img, cmap='gray', vmin=-1, vmax=1)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Arquitectura DCGAN para imágenes 64×64

### Diagrama de la arquitectura

```
GENERADOR
z (100) → Linear → (512, 4, 4) → ConvT 256 → ConvT 128 → ConvT 64 → ConvT 1 → imagen (1, 64, 64)
                                   (8×8)       (16×16)     (32×32)    (64×64)

DISCRIMINADOR
imagen (1, 64, 64) → Conv 64 → Conv 128 → Conv 256 → Conv 512 → Linear → prob real/falso
                     (32×32)   (16×16)    (8×8)      (4×4)
```

Cada bloque convolucional transpuesto dobla la resolución espacial (stride=2).

### 5.1 Generador

In [ ]:
class Generador(nn.Module):
    """
    DCGAN Generador para imágenes de 64×64 en escala de grises.
    
    Entrada : vector z de dimensión Z_DIM (ruido gaussiano)
    Salida  : imagen (1, 64, 64) con valores en [-1, 1]  (Tanh)
    
    Progresión de resoluciones:
        4×4 → 8×8 → 16×16 → 32×32 → 64×64
    """
    
    def __init__(self, z_dim=100, canales_base=64):
        super().__init__()
        self.input_size = z_dim    # alias para compatibilidad con el loop fit()
        self.z_dim = z_dim
        cb = canales_base          # 64 por defecto
        
        # Proyección del vector z a mapa de características 4×4
        self.proyeccion = nn.Sequential(
            nn.Linear(z_dim, 8 * cb * 4 * 4),   # z → (512, 4, 4)
            nn.BatchNorm1d(8 * cb * 4 * 4),
            nn.ReLU(True)
        )
        
        # Bloques convolucionales transpuestos (upsamppling)
        self.conv_transpuesta = nn.Sequential(
            # (512, 4, 4) → (256, 8, 8)
            nn.ConvTranspose2d(8*cb, 4*cb, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(4*cb),
            nn.ReLU(True),
            
            # (256, 8, 8) → (128, 16, 16)
            nn.ConvTranspose2d(4*cb, 2*cb, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(2*cb),
            nn.ReLU(True),
            
            # (128, 16, 16) → (64, 32, 32)
            nn.ConvTranspose2d(2*cb, cb, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(cb),
            nn.ReLU(True),
            
            # (64, 32, 32) → (1, 64, 64)
            nn.ConvTranspose2d(cb, 1, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()  # salida en [-1, 1]
        )
    
    def forward(self, z):
        # z: (B, z_dim)
        x = self.proyeccion(z)                   # (B, 512*4*4)
        x = x.view(-1, 8*64, 4, 4)              # (B, 512, 4, 4)
        x = self.conv_transpuesta(x)             # (B, 1, 64, 64)
        return x.view(x.size(0), -1)             # aplanar → (B, 64*64=4096) para el loop fit()


# Probar el generador
gen = Generador(z_dim=100, canales_base=64)
z_prueba = torch.randn(4, 100)
salida   = gen(z_prueba)
print(f"Salida del generador: {salida.shape}")
print(f"Salida como imagen   : {salida.view(4, 1, 64, 64).shape}")
print(f"Parámetros           : {sum(p.numel() for p in gen.parameters()):,}")

### 5.2 Discriminador

In [ ]:
class Discriminador(nn.Module):
    """
    DCGAN Discriminador para imágenes de 64×64 en escala de grises.
    
    Entrada : vector aplanado (4096,) o imagen (1, 64, 64)
    Salida  : probabilidad de ser real  ∈ [0, 1]  (Sigmoid)
    
    Progresión de resoluciones (downsampling):
        64×64 → 32×32 → 16×16 → 8×8 → 4×4
    """
    
    def __init__(self, canales_base=64):
        super().__init__()
        cb = canales_base
        
        # Bloques convolucionales (downsampling)
        self.conv = nn.Sequential(
            # (1, 64, 64) → (64, 32, 32)
            nn.Conv2d(1, cb, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            # (64, 32, 32) → (128, 16, 16)
            nn.Conv2d(cb, 2*cb, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(2*cb),
            nn.LeakyReLU(0.2, inplace=True),
            
            # (128, 16, 16) → (256, 8, 8)
            nn.Conv2d(2*cb, 4*cb, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(4*cb),
            nn.LeakyReLU(0.2, inplace=True),
            
            # (256, 8, 8) → (512, 4, 4)
            nn.Conv2d(4*cb, 8*cb, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(8*cb),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # Clasificador final
        self.clasificador = nn.Sequential(
            nn.Linear(8*cb * 4 * 4, 1),  # 512*4*4 = 8192 → 1
            nn.Sigmoid()
        )
    
    def forward(self, x):
        # x puede venir aplanado (B, 4096) del loop fit()
        if x.dim() == 2:
            x = x.view(x.size(0), 1, 64, 64)
        x = self.conv(x)                   # (B, 512, 4, 4)
        x = x.view(x.size(0), -1)          # (B, 8192)
        return self.clasificador(x)         # (B, 1)


# Probar el discriminador
disc = Discriminador(canales_base=64)
img_prueba = torch.randn(4, 1, 64, 64)
prob = disc(img_prueba)
print(f"Entrada discriminador: {img_prueba.shape}")
print(f"Probabilidad de salida: {prob.shape}")
print(f"Parámetros: {sum(p.numel() for p in disc.parameters()):,}")

## 6. Función de entrenamiento (compatible con el cuadernillo base)

El bucle de entrenamiento sigue la misma lógica del notebook base (`01_gans.ipynb`):

1. **Fase del Discriminador:** batch real (etiqueta=1) + imágenes generadas (etiqueta=0) → BCE loss
2. **Fase del Generador:** genera imágenes → el discriminador las evalúa → BCE loss contra etiqueta=1

In [ ]:
def fit(g, d, dataloader, epochs=50, lr=2e-4, crit=None, mostrar_cada=10):
    """
    Entrena la DCGAN.
    
    Parámetros
    ----------
    g             : módulo Generador
    d             : módulo Discriminador  
    dataloader    : DataLoader con las imágenes reales
    epochs        : número de épocas
    lr            : learning rate para Adam
    crit          : función de pérdida (default: BCELoss)
    mostrar_cada  : cada cuántas épocas mostrar imágenes generadas
    
    Retorna
    -------
    hist : dict con listas 'g_loss' y 'd_loss' por época
    """
    g.to(device)
    d.to(device)
    
    opt_g = torch.optim.Adam(g.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(d.parameters(), lr=lr, betas=(0.5, 0.999))
    
    if crit is None:
        crit = nn.BCELoss()
    
    # Vector de ruido fijo para seguimiento visual durante el entrenamiento
    z_fijo = torch.randn(16, g.input_size, device=device)
    
    hist = {'g_loss': [], 'd_loss': []}
    
    for epoch in range(1, epochs + 1):
        g_losses, d_losses = [], []
        
        for X, _ in dataloader:
            bs = X.size(0)
            
            # ── FASE 1: Entrenar Discriminador ──────────────────────────────
            g.eval(); d.train()
            
            # Ruido → imágenes falsas
            z = torch.randn(bs, g.input_size, device=device)
            imgs_falsas = g(z).detach()          # no propagar gradientes al generador
            
            # Concatenar reales + falsas
            d_input   = torch.cat([imgs_falsas, X.view(bs, -1)])
            d_etiquetas = torch.cat([
                torch.zeros(bs, 1, device=device),   # falsas → 0
                torch.ones(bs,  1, device=device)    # reales  → 1
            ])
            
            opt_d.zero_grad()
            d_output = d(d_input)
            d_l      = crit(d_output, d_etiquetas)
            d_l.backward()
            opt_d.step()
            d_losses.append(d_l.item())
            
            # ── FASE 2: Entrenar Generador ───────────────────────────────────
            g.train(); d.eval()
            
            z = torch.randn(bs, g.input_size, device=device)
            imgs_gen = g(z)
            d_pred   = d(imgs_gen)
            # El generador quiere engañar al discriminador → etiqueta real (1)
            g_etiquetas = torch.ones(bs, 1, device=device)
            
            opt_g.zero_grad()
            g_l = crit(d_pred, g_etiquetas)
            g_l.backward()
            opt_g.step()
            g_losses.append(g_l.item())
        
        # Métricas de época
        g_mean = np.mean(g_losses)
        d_mean = np.mean(d_losses)
        hist['g_loss'].append(g_mean)
        hist['d_loss'].append(d_mean)
        print(f"Época {epoch:3d}/{epochs} | g_loss={g_mean:.4f}  d_loss={d_mean:.4f}")
        
        # Visualizar progreso cada N épocas
        if epoch % mostrar_cada == 0:
            g.eval()
            with torch.no_grad():
                imgs = g(z_fijo).view(-1, 1, 64, 64).cpu()
            grid = vutils.make_grid(imgs, nrow=4, normalize=True, value_range=(-1,1))
            plt.figure(figsize=(8, 8))
            plt.imshow(grid.permute(1,2,0).numpy(), cmap='gray')
            plt.title(f'Imágenes generadas — Época {epoch}', fontsize=13)
            plt.axis('off')
            plt.show()
    
    return hist

## 7. Inicializar y entrenar la DCGAN

In [ ]:
# ── Parámetros de entrenamiento ───────────────────────────────────────────────
Z_DIM       = 100    # dimensión del vector de ruido latente
CANALES_BASE = 64    # base para el escalado de canales (64→128→256→512)
EPOCHS      = 50     # número de épocas (aumentar para mejores resultados)
LR          = 2e-4   # learning rate (recomendado para DCGANs)
MOSTRAR_CADA = 10    # mostrar imágenes cada N épocas
# ─────────────────────────────────────────────────────────────────────────────

# Crear modelos
generador     = Generador(z_dim=Z_DIM, canales_base=CANALES_BASE)
discriminador = Discriminador(canales_base=CANALES_BASE)

print("=" * 55)
print(f"  GENERADOR     — {sum(p.numel() for p in generador.parameters()):>10,} parámetros")
print(f"  DISCRIMINADOR — {sum(p.numel() for p in discriminador.parameters()):>10,} parámetros")
print("=" * 55)
print(f"  Dataset: {len(dataset)} imágenes | Batch: {BATCH_SIZE} | Épocas: {EPOCHS}")
print(f"  Dispositivo: {device}")
print("=" * 55)

In [ ]:
# Verificar imagen antes de entrenar (generador sin entrenar)
generador.eval()
with torch.no_grad():
    z_test = torch.randn(8, Z_DIM)
    imgs_test = generador(z_test).view(8, 1, 64, 64).cpu()

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
fig.suptitle('Imágenes ANTES del entrenamiento (ruido puro)', fontsize=12)
for ax, img in zip(axes.flatten(), imgs_test):
    ax.imshow(img[0].numpy(), cmap='gray', vmin=-1, vmax=1)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── ENTRENAMIENTO ─────────────────────────────────────────────────────────────
hist = fit(
    generador,
    discriminador,
    dataloader,
    epochs=EPOCHS,
    lr=LR,
    crit=nn.BCELoss(),
    mostrar_cada=MOSTRAR_CADA
)

## 8. Análisis de las curvas de pérdida

> **Interpretación:** En una GAN bien entrenada:
> - La `d_loss` (discriminador) tiende hacia **0.5** (no puede distinguir reales de falsos)
> - La `g_loss` (generador) disminuye indicando que engaña mejor al discriminador
> - Si la `d_loss` → 0: el discriminador domina (el generador no aprende)
> - Si la `g_loss` → 0: el generador colapsa (mode collapse)

In [ ]:
df_hist = pd.DataFrame(hist)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pérdidas por separado
axes[0].plot(df_hist['g_loss'], label='Generador',    color='steelblue',  lw=2)
axes[0].plot(df_hist['d_loss'], label='Discriminador',color='darkorange', lw=2)
axes[0].axhline(0.5, ls='--', color='gray', alpha=0.7, label='Equilibrio (0.5)')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Pérdida (BCE)')
axes[0].set_title('Curvas de pérdida — DCGAN')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Ratio G/D (indica equilibrio)
ratio = df_hist['g_loss'] / (df_hist['d_loss'] + 1e-8)
axes[1].plot(ratio, color='purple', lw=2)
axes[1].axhline(1.0, ls='--', color='gray', alpha=0.7, label='Ratio ideal = 1')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('g_loss / d_loss')
axes[1].set_title('Ratio de pérdidas (equilibrio GAN)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nÚltima época — g_loss: {hist['g_loss'][-1]:.4f} | d_loss: {hist['d_loss'][-1]:.4f}")

## 9. Generación de nuevas imágenes

In [ ]:
def generar_imagenes(generador, n=20, z_dim=100, titulo='Imágenes generadas por la DCGAN'):
    """Genera y muestra n imágenes nuevas del generador entrenado."""
    generador.eval()
    with torch.no_grad():
        z = torch.randn(n, z_dim, device=device)
        imgs = generador(z).view(n, 1, 64, 64).cpu()
    
    cols = 5
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
    fig.suptitle(titulo, fontsize=14, y=1.01)
    
    for i, ax in enumerate(axes.flatten()):
        if i < n:
            ax.imshow(imgs[i, 0].numpy(), cmap='gray', vmin=-1, vmax=1)
            ax.set_title(f'#{i+1}', fontsize=9)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    return imgs

imgs_nuevas = generar_imagenes(generador, n=20, z_dim=Z_DIM)

In [ ]:
# Comparación: imágenes reales vs. generadas
n_compare = 8

# Tomar muestras reales
idx_reales = np.random.choice(len(dataset), n_compare, replace=False)
reales = torch.stack([dataset[i][0] for i in idx_reales]).view(n_compare, 1, 64, 64).cpu()

# Generar falsas
generador.eval()
with torch.no_grad():
    z = torch.randn(n_compare, Z_DIM, device=device)
    falsas = generador(z).view(n_compare, 1, 64, 64).cpu()

fig, axes = plt.subplots(2, n_compare, figsize=(n_compare*2.5, 6))
fig.suptitle('Comparación: Reales vs. Generadas', fontsize=14)

for i in range(n_compare):
    axes[0, i].imshow(reales[i, 0].numpy(), cmap='gray', vmin=-1, vmax=1)
    axes[0, i].axis('off')
    if i == 0: axes[0, i].set_title('REALES', fontsize=10, fontweight='bold')
    
    axes[1, i].imshow(falsas[i, 0].numpy(), cmap='gray', vmin=-1, vmax=1)
    axes[1, i].axis('off')
    if i == 0: axes[1, i].set_title('GENERADAS', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Guardar el modelo entrenado

In [ ]:
# Guardar pesos del generador y discriminador
ruta_guardado = r'D:\2026\IA-2026-DATASETS\animacion_dataset'
os.makedirs(ruta_guardado, exist_ok=True)

ruta_gen  = os.path.join(ruta_guardado, 'dcgan_generador.pth')
ruta_disc = os.path.join(ruta_guardado, 'dcgan_discriminador.pth')

torch.save(generador.state_dict(),     ruta_gen)
torch.save(discriminador.state_dict(), ruta_disc)

print(f"✓ Generador guardado     → {ruta_gen}")
print(f"✓ Discriminador guardado → {ruta_disc}")

## 11. (Opcional) Exploración del espacio latente

La interpolación en el espacio latente muestra cómo el generador interpola suavemente entre imágenes.

In [ ]:
def interpolar_latente(generador, z_dim=100, n_pasos=10):
    """Interpola linealmente entre dos vectores z aleatorios."""
    generador.eval()
    with torch.no_grad():
        z1 = torch.randn(1, z_dim, device=device)
        z2 = torch.randn(1, z_dim, device=device)
        
        alphas = np.linspace(0, 1, n_pasos)
        imgs_interp = []
        for a in alphas:
            z_interp = (1 - a) * z1 + a * z2
            img = generador(z_interp).view(1, 64, 64).cpu().numpy()
            imgs_interp.append(img[0])
    
    fig, axes = plt.subplots(1, n_pasos, figsize=(n_pasos*2, 2.5))
    fig.suptitle(f'Interpolación en el espacio latente (z₁ → z₂, {n_pasos} pasos)', fontsize=12)
    for ax, img, a in zip(axes, imgs_interp, alphas):
        ax.imshow(img, cmap='gray', vmin=-1, vmax=1)
        ax.set_title(f'α={a:.1f}', fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

interpolar_latente(generador, z_dim=Z_DIM, n_pasos=10)

## 12. Resumen

En este cuadernillo hemos implementado una **DCGAN** adaptada al dataset multimodal de animación:

### ¿Qué hicimos?

| Sección | Descripción |
|---------|-------------|
| **Exploración** | Analizamos las 3 modalidades del dataset: tabular (CSV), imágenes (.npy 64×64) y audio (.wav) |
| **Dataset** | Creamos un `DatasetAnimacion` que carga y normaliza las imágenes de [-1,1] para la activación Tanh |
| **Arquitectura** | Generador con 4 capas `ConvTranspose2d` (4×4 → 64×64) + Discriminador con 4 capas `Conv2d` |
| **Entrenamiento** | Bucle de dos fases (discriminador → generador) con `BCELoss` y `Adam` (lr=2e-4, betas=(0.5,0.999)) |
| **Evaluación** | Curvas de pérdida, comparación real vs. generado, interpolación en el espacio latente |

### Diferencias clave vs. GAN simple (MLP)

- **Imágenes 64×64** en lugar de 28×28 (Fashion MNIST)
- **Convoluciones** en lugar de capas lineales → captura estructura espacial
- **BatchNorm2d** y **LeakyReLU** en el discriminador → entrenamiento más estable
- **Dataset multimodal real** con imágenes de storyboard de animación

### Posibles mejoras

1. Aumentar épocas (100-200) para mejores resultados
2. Experimentar con **WGAN-GP** (Wasserstein GAN con penalización de gradiente) para entrenamiento más estable
3. Incorporar la modalidad de **audio** generando espectrogramas
4. Usar **información tabular** como condicionamiento (cGAN) para controlar los atributos de las imágenes generadas